# Задача №4. Пайплайн прогнозирования временного ряда

- статистическая модель из задачи №2: `SeasonalES`
- data-driven модель из задачи №3: `LightGBM`


In [1]:
from pathlib import Path
import time, json, warnings, math
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from lightgbm import LGBMRegressor
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.stattools import adfuller
from scipy.stats import jarque_bera, shapiro, wilcoxon

try:
    from scipy.stats import median_abs_deviation
except Exception:
    median_abs_deviation = None

BASE = Path.cwd()
if not (BASE / 'powerconsumption.csv').exists():
    BASE = Path('/mnt/data')
DATA_PATH = BASE / 'powerconsumption.csv'
OUT = BASE / 'task4_outputs'
FIG = OUT / 'figures'
OUT.mkdir(exist_ok=True)
FIG.mkdir(exist_ok=True)

TARGET = 'PowerConsumption_Zone1'
H = 24
SEASON_LENGTH = 24
MAX_TRAIN_SIZE = 24 * 90
N_WINDOWS = 3


def mape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = np.where(y_true == 0, np.nan, y_true)
    return float(np.nanmean(np.abs((y_true - y_pred) / denom)) * 100)

def smape(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denom = (np.abs(y_true) + np.abs(y_pred)) / 2
    denom = np.where(denom == 0, np.nan, denom)
    return float(np.nanmean(np.abs(y_true - y_pred) / denom) * 100)

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = math.sqrt(mean_squared_error(y_true, y_pred))
    return {
        'MAE': float(mae),
        'RMSE': float(rmse),
        'MAPE_%': mape(y_true, y_pred),
        'sMAPE_%': smape(y_true, y_pred),
        'bias': float(np.mean(y_pred - y_true)),
    }

# 1. Load and prepare
raw = pd.read_csv(DATA_PATH)
raw['Datetime'] = pd.to_datetime(raw['Datetime'], errors='coerce')
raw = raw.dropna(subset=['Datetime']).drop_duplicates().sort_values('Datetime')

hourly = (
    raw[['Datetime', TARGET]]
    .set_index('Datetime')
    .resample('H').mean()
    .rename(columns={TARGET:'y'})
    .reset_index()
    .rename(columns={'Datetime':'ds'})
)
hourly['unique_id'] = 'zone1'
hourly = hourly[['unique_id','ds','y']].dropna().reset_index(drop=True)

# Checks
quality = pd.DataFrame({
    'check': ['rows_10min_raw','rows_hourly','missing_y_hourly','duplicates_raw_after_clean','freq_hourly_regular'],
    'value': [len(raw), len(hourly), int(hourly['y'].isna().sum()), int(raw.duplicated().sum()), bool((hourly['ds'].diff().dropna() == pd.Timedelta(hours=1)).all())]
})
quality.to_csv(OUT/'data_quality_checks.csv', index=False)

# EDA/per-model plots
plt.figure(figsize=(14,4))
plt.plot(hourly['ds'], hourly['y'], linewidth=0.8)
plt.title('Почасовой ряд PowerConsumption_Zone1')
plt.xlabel('Дата'); plt.ylabel('Потребление')
plt.tight_layout(); plt.savefig(FIG/'01_hourly_series.png', dpi=160); plt.close()

# feature engineering
LAGS = [1,2,3,24,25,48,72,168]
ROLLS = [24,168]

def add_time_features(df):
    out = df.copy()
    ds = pd.to_datetime(out['ds'])
    out['hour'] = ds.dt.hour
    out['dayofweek'] = ds.dt.dayofweek
    out['month'] = ds.dt.month
    out['is_weekend'] = (out['dayofweek'] >= 5).astype(int)
    out['hour_sin'] = np.sin(2*np.pi*out['hour']/24)
    out['hour_cos'] = np.cos(2*np.pi*out['hour']/24)
    out['dow_sin'] = np.sin(2*np.pi*out['dayofweek']/7)
    out['dow_cos'] = np.cos(2*np.pi*out['dayofweek']/7)
    return out

def make_features(df):
    out = add_time_features(df[['ds','y']].copy())
    for lag in LAGS:
        out[f'lag_{lag}'] = out['y'].shift(lag)
    shifted = out['y'].shift(1)
    for w in ROLLS:
        out[f'roll_mean_{w}'] = shifted.rolling(w).mean()
        out[f'roll_std_{w}'] = shifted.rolling(w).std()
    out['diff_1'] = out['lag_1'] - out['lag_2']
    out['diff_24'] = out['lag_24'] - out['lag_48']
    return out.dropna().reset_index(drop=True)

FEATURE_COLS = ['hour','dayofweek','month','is_weekend','hour_sin','hour_cos','dow_sin','dow_cos'] + \
               [f'lag_{l}' for l in LAGS] + [f'roll_mean_{w}' for w in ROLLS] + [f'roll_std_{w}' for w in ROLLS] + ['diff_1','diff_24']

def recursive_predict(model, train_df, horizon):
    hist = train_df[['ds','y']].copy().reset_index(drop=True)
    preds = []
    start = hist['ds'].iloc[-1]
    for step in range(1, horizon+1):
        next_ds = start + pd.Timedelta(hours=step)
        tmp = pd.concat([hist, pd.DataFrame({'ds':[next_ds], 'y':[np.nan]})], ignore_index=True)
        row = add_time_features(tmp.tail(1))[['ds','y','hour','dayofweek','month','is_weekend','hour_sin','hour_cos','dow_sin','dow_cos']].copy()
        # build lag/rolling from hist y
        yhist = hist['y'].reset_index(drop=True)
        for lag in LAGS:
            row[f'lag_{lag}'] = yhist.iloc[-lag] if len(yhist) >= lag else np.nan
        for w in ROLLS:
            vals = yhist.iloc[-w:]
            row[f'roll_mean_{w}'] = vals.mean() if len(vals) >= w else np.nan
            row[f'roll_std_{w}'] = vals.std(ddof=1) if len(vals) >= w else np.nan
        row['diff_1'] = row['lag_1'] - row['lag_2']
        row['diff_24'] = row['lag_24'] - row['lag_48']
        X = row[FEATURE_COLS]
        pred = float(model.predict(X)[0])
        preds.append({'ds': next_ds, 'y_pred': pred})
        hist = pd.concat([hist, pd.DataFrame({'ds':[next_ds], 'y':[pred]})], ignore_index=True)
    return pd.DataFrame(preds)

def fit_predict_model(model_name, train_df, horizon):
    start_fit = time.perf_counter()
    if model_name == 'SeasonalNaive':
        model_obj = None
        fit_sec = time.perf_counter() - start_fit
        start_pred = time.perf_counter()
        last_season = train_df['y'].iloc[-SEASON_LENGTH:].to_numpy()
        yhat = np.resize(last_season, horizon)
        pred = pd.DataFrame({'ds': pd.date_range(train_df['ds'].iloc[-1] + pd.Timedelta(hours=1), periods=horizon, freq='H'), 'y_pred': yhat})
        pred_sec = time.perf_counter() - start_pred
        return pred, fit_sec, pred_sec
    if model_name == 'SeasonalES':
        # Seasonal exponential smoothing, alpha=0.8, matching the Task 2 selected manual SeasonalES setup.
        alpha = 0.8
        vals = train_df['y'].astype(float).to_numpy()
        states = vals[:SEASON_LENGTH].astype(float).copy()
        for i, value in enumerate(vals[SEASON_LENGTH:], start=SEASON_LENGTH):
            pos = i % SEASON_LENGTH
            states[pos] = alpha * value + (1 - alpha) * states[pos]
        fit_sec = time.perf_counter() - start_fit
        start_pred = time.perf_counter()
        start_pos = len(vals)
        yhat = np.array([states[(start_pos + i) % SEASON_LENGTH] for i in range(horizon)])
        pred = pd.DataFrame({'ds': pd.date_range(train_df['ds'].iloc[-1] + pd.Timedelta(hours=1), periods=horizon, freq='H'), 'y_pred': yhat})
        pred_sec = time.perf_counter() - start_pred
        return pred, fit_sec, pred_sec
    # ML models
    feat = make_features(train_df)
    X_train = feat[FEATURE_COLS]
    y_train = feat['y']
    if model_name == 'LinearRegression':
        model = LinearRegression()
    elif model_name == 'RandomForest':
        model = RandomForestRegressor(n_estimators=50, min_samples_leaf=2, random_state=42, n_jobs=1)
    elif model_name == 'LightGBM':
        model = LGBMRegressor(n_estimators=200, learning_rate=0.05, num_leaves=31, random_state=42, verbosity=-1, n_jobs=1)
    else:
        raise ValueError(model_name)
    model.fit(X_train, y_train)
    fit_sec = time.perf_counter() - start_fit
    start_pred = time.perf_counter()
    pred = recursive_predict(model, train_df, horizon)
    pred_sec = time.perf_counter() - start_pred
    return pred, fit_sec, pred_sec

CANDIDATE_MODELS = ['SeasonalNaive','SeasonalES','LinearRegression','RandomForest','LightGBM']
# backtesting: last N windows with h steps
all_rows = []
all_preds = []
start_last_cutoff = len(hourly) - H * N_WINDOWS
for w in range(N_WINDOWS):
    test_start = start_last_cutoff + w*H
    test_end = test_start + H
    train_start = max(0, test_start - MAX_TRAIN_SIZE)
    train_w = hourly.iloc[train_start:test_start].copy().reset_index(drop=True)
    test_w = hourly.iloc[test_start:test_end].copy().reset_index(drop=True)
    for model_name in CANDIDATE_MODELS:
        pred, fit_sec, pred_sec = fit_predict_model(model_name, train_w, H)
        yy = test_w['y'].to_numpy()
        row = {'window': w+1, 'model': model_name, 'train_rows': len(train_w), 'test_rows': len(test_w), 'fit_sec': fit_sec, 'predict_sec': pred_sec}
        row.update(metrics(yy, pred['y_pred'].to_numpy()))
        all_rows.append(row)
        pp = pred.copy()
        pp['window'] = w+1
        pp['model'] = model_name
        pp['y'] = yy
        all_preds.append(pp)

bt = pd.DataFrame(all_rows)
preds = pd.concat(all_preds, ignore_index=True)
bt.to_csv(OUT/'backtesting_by_window.csv', index=False)
preds.to_csv(OUT/'backtesting_predictions.csv', index=False)
summary = (bt.groupby('model')
             .agg(MAE=('MAE','mean'), RMSE=('RMSE','mean'), MAPE_pct=('MAPE_%','mean'), sMAPE_pct=('sMAPE_%','mean'), bias=('bias','mean'), fit_sec_mean=('fit_sec','mean'), predict_sec_mean=('predict_sec','mean'))
             .reset_index()
             .sort_values('RMSE'))
summary.to_csv(OUT/'model_comparison_summary.csv', index=False)

# hold-out (last day), train previous max train size
train_final = hourly.iloc[-H-MAX_TRAIN_SIZE:-H].copy().reset_index(drop=True)
test_final = hourly.iloc[-H:].copy().reset_index(drop=True)
hold_rows = []
hold_pred_frames = []
for model_name in CANDIDATE_MODELS:
    pred, fit_sec, pred_sec = fit_predict_model(model_name, train_final, H)
    row = {'model': model_name, 'fit_sec': fit_sec, 'predict_sec': pred_sec}
    row.update(metrics(test_final['y'], pred['y_pred']))
    hold_rows.append(row)
    p = pred.copy(); p['model'] = model_name; p['y'] = test_final['y'].to_numpy()
    hold_pred_frames.append(p)
holdout = pd.DataFrame(hold_rows).sort_values('RMSE').reset_index(drop=True)
holdout.to_csv(OUT/'holdout_metrics.csv', index=False)
hold_preds = pd.concat(hold_pred_frames, ignore_index=True)
hold_preds.to_csv(OUT/'holdout_predictions.csv', index=False)

# Choose final model: main data-driven candidate if it wins backtest; else best by RMSE
best_model = summary.iloc[0]['model']
# if tied very close choose faster? no
final_pred = hold_preds[hold_preds['model']==best_model].copy()
resid = final_pred['y'].to_numpy() - final_pred['y_pred'].to_numpy()

# residual tests
resid_tests = []
try:
    lb = acorr_ljungbox(resid, lags=[min(10, len(resid)//2)], return_df=True)
    resid_tests.append({'test':'Ljung-Box residual autocorrelation', 'stat':float(lb['lb_stat'].iloc[0]), 'p_value':float(lb['lb_pvalue'].iloc[0]), 'interpretation': 'p<0.05: у остатков есть статистически значимая автокорреляция; p>0.05: автокорреляция не выявлена'})
except Exception as e:
    resid_tests.append({'test':'Ljung-Box residual autocorrelation', 'stat':np.nan, 'p_value':np.nan, 'interpretation':f'не рассчитано: {e}'})
try:
    jb = jarque_bera(resid)
    resid_tests.append({'test':'Jarque-Bera residual normality', 'stat':float(jb.statistic), 'p_value':float(jb.pvalue), 'interpretation': 'p>0.05: нормальность остатков не отвергается'})
except Exception as e:
    resid_tests.append({'test':'Jarque-Bera residual normality', 'stat':np.nan, 'p_value':np.nan, 'interpretation':f'не рассчитано: {e}'})
try:
    adf = adfuller(pd.Series(resid).dropna(), autolag='AIC')
    resid_tests.append({'test':'ADF residual stationarity', 'stat':float(adf[0]), 'p_value':float(adf[1]), 'interpretation': 'p<0.05: остатки стационарны; p>=0.05: стационарность не подтверждена'})
except Exception as e:
    resid_tests.append({'test':'ADF residual stationarity', 'stat':np.nan, 'p_value':np.nan, 'interpretation':f'не рассчитано: {e}'})

# pairwise Wilcoxon abs errors final vs main stat/data candidates if possible
# DM-like simple test: Wilcoxon paired absolute errors on all backtest observations
pivot_pred = preds.pivot_table(index=['window','ds'], columns='model', values=['y_pred','y'])
# y same for models; build pair data
try:
    y_vals = preds.drop_duplicates(['window','ds']).sort_values(['window','ds'])['y'].to_numpy()
    best_err = np.abs(y_vals - preds[preds['model']==best_model].sort_values(['window','ds'])['y_pred'].to_numpy())
    for other in ['SeasonalES','LightGBM','SeasonalNaive']:
        if other != best_model and other in preds['model'].unique():
            other_err = np.abs(y_vals - preds[preds['model']==other].sort_values(['window','ds'])['y_pred'].to_numpy())
            st = wilcoxon(best_err, other_err, alternative='less')
            resid_tests.append({'test':f'Wilcoxon |error|: {best_model} < {other}', 'stat':float(st.statistic), 'p_value':float(st.pvalue), 'interpretation': 'p<0.05: ошибка выбранной модели статистически ниже'})
except Exception as e:
    resid_tests.append({'test':'Wilcoxon paired errors', 'stat':np.nan, 'p_value':np.nan, 'interpretation':f'не рассчитано: {e}'})

pd.DataFrame(resid_tests).to_csv(OUT/'statistical_tests.csv', index=False)

# Plots
plt.figure(figsize=(10,4))
plot_df = summary.sort_values('RMSE')
plt.bar(plot_df['model'], plot_df['RMSE'])
plt.xticks(rotation=30, ha='right')
plt.title('Backtesting: средний RMSE по 5 окнам')
plt.ylabel('RMSE')
plt.tight_layout(); plt.savefig(FIG/'02_backtesting_rmse.png', dpi=160); plt.close()

plt.figure(figsize=(10,4))
plot_df = summary.sort_values('fit_sec_mean')
plt.bar(plot_df['model'], plot_df['fit_sec_mean'])
plt.xticks(rotation=30, ha='right')
plt.title('Среднее время обучения на окно')
plt.ylabel('секунды')
plt.tight_layout(); plt.savefig(FIG/'03_fit_time.png', dpi=160); plt.close()

plt.figure(figsize=(14,5))
plt.plot(train_final['ds'].tail(7*24), train_final['y'].tail(7*24), label='train tail')
plt.plot(test_final['ds'], test_final['y'], label='actual')
for model_name in ['SeasonalES','LightGBM',best_model]:
    if model_name in hold_preds['model'].unique():
        p = hold_preds[hold_preds['model']==model_name]
        lw = 2.5 if model_name==best_model else 1.2
        plt.plot(p['ds'], p['y_pred'], label=f'forecast {model_name}', linewidth=lw)
plt.title(f'Hold-out прогноз на 24 часа, финальная модель: {best_model}')
plt.xlabel('Дата'); plt.ylabel('Потребление')
plt.legend()
plt.tight_layout(); plt.savefig(FIG/'04_holdout_forecast.png', dpi=160); plt.close()

plt.figure(figsize=(14,4))
plt.plot(final_pred['ds'], resid, marker='o')
plt.axhline(0, linewidth=1)
plt.title(f'Остатки финальной модели: {best_model}')
plt.xlabel('Дата'); plt.ylabel('y - y_pred')
plt.tight_layout(); plt.savefig(FIG/'05_residuals_final.png', dpi=160); plt.close()

# Anomaly detection (for pipeline robustness)
series = hourly['y']
roll_mean = series.rolling(24).mean()
roll_std = series.rolling(24).std()
z = (series - roll_mean) / roll_std.replace(0, np.nan)
anom_roll = z.abs() > 3
# robust residual from daily median pattern
profile = hourly.assign(hour=hourly['ds'].dt.hour).groupby('hour')['y'].median()
seasonal_est = hourly['ds'].dt.hour.map(profile).astype(float)
resid_season = series - seasonal_est
mad = float(np.median(np.abs(resid_season - np.median(resid_season))))
robust_z = 0.6745*(resid_season - np.median(resid_season))/mad if mad>0 else pd.Series(np.zeros(len(series)))
anom_robust = np.abs(robust_z) > 3.5
anom_summary = pd.DataFrame({
    'method':['Rolling z-score 24h','Daily-profile robust z-score'],
    'n_anomalies':[int(anom_roll.sum()), int(np.sum(anom_robust))],
    'share_%':[float(anom_roll.mean()*100), float(np.mean(anom_robust)*100)]
})
anom_summary.to_csv(OUT/'anomaly_checks.csv', index=False)

viz = hourly.tail(24*14).copy().reset_index(drop=True)
mask_roll = anom_roll.tail(24*14).reset_index(drop=True)
mask_rob = pd.Series(anom_robust).tail(24*14).reset_index(drop=True)
plt.figure(figsize=(14,5))
plt.plot(viz['ds'], viz['y'], label='y')
plt.scatter(viz.loc[mask_roll, 'ds'], viz.loc[mask_roll, 'y'], marker='o', label='rolling z')
plt.scatter(viz.loc[mask_rob, 'ds'], viz.loc[mask_rob, 'y'], marker='x', label='robust z')
plt.title('Проверка аномалий на последних 14 днях')
plt.xlabel('Дата'); plt.ylabel('Потребление')
plt.legend()
plt.tight_layout(); plt.savefig(FIG/'06_anomaly_checks.png', dpi=160); plt.close()

# Save config/summary JSON
config = {
    'target': TARGET,
    'resample_frequency':'1H mean',
    'horizon_hours': H,
    'season_length_hours': SEASON_LENGTH,
    'n_backtesting_windows': N_WINDOWS,
    'max_train_size_hours': MAX_TRAIN_SIZE,
    'candidate_models': CANDIDATE_MODELS,
    'selected_from_task2':'SeasonalES',
    'selected_from_task3':'LightGBM',
    'final_best_by_backtesting_rmse': best_model,
    'data_start': str(hourly['ds'].min()),
    'data_end': str(hourly['ds'].max()),
}
with open(OUT/'pipeline_config.json','w',encoding='utf-8') as f:
    json.dump(config,f,ensure_ascii=False,indent=2)

print('DONE')
print(summary)
print(holdout)
print(pd.DataFrame(resid_tests))
print(anom_summary)


DONE
              model          MAE         RMSE  MAPE_pct  sMAPE_pct  \
0          LightGBM   620.934188   838.159118  2.027955   2.029759   
2      RandomForest   707.538543   987.063769  2.282921   2.265817   
1  LinearRegression   751.137528   998.671774  2.495434   2.469202   
4     SeasonalNaive  1001.126602  1275.124646  3.373083   3.306671   
3        SeasonalES  1030.491700  1295.285975  3.455155   3.383009   

         bias  fit_sec_mean  predict_sec_mean  
0  -44.703099  1.942434e-01          0.181103  
2  162.975689  9.207644e-01          0.232595  
1  234.039313  1.491567e-02          0.194305  
4  607.097592  2.666493e-07          0.000637  
3  649.594396  7.351333e-04          0.000649  
              model       fit_sec  predict_sec          MAE         RMSE  \
0          LightGBM  1.861667e-01     0.177273   680.912127   828.389594   
1      RandomForest  8.977416e-01     0.240511   904.709220  1090.659472   
2  LinearRegression  1.368890e-02     0.179819  1013.68620